In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv('../data/dataset.csv')

# Get all symptom columns
symptom_cols = [c for c in df.columns if 'Symptom' in c]

# Get all unique symptoms
all_symptoms = set()
for col in symptom_cols:
    all_symptoms.update(df[col].dropna().str.strip().unique())

all_symptoms = sorted(list(all_symptoms))
print(f"Total unique symptoms: {len(all_symptoms)}")
print(f"Total diseases: {df['Disease'].nunique()}")
print(f"Total rows: {len(df)}")

Total unique symptoms: 131
Total diseases: 41
Total rows: 4920


In [2]:
# Create binary matrix — 1 if symptom present, 0 if not
# This is the KEY preprocessing step

def create_feature_matrix(df, all_symptoms):
    X = pd.DataFrame(0, index=df.index, columns=all_symptoms)
    
    for col in symptom_cols:
        for idx, symptom in df[col].dropna().items():
            symptom = symptom.strip()
            if symptom in X.columns:
                X.loc[idx, symptom] = 1
    
    return X

X = create_feature_matrix(df, all_symptoms)
y = df['Disease']

print("Feature matrix shape:", X.shape)
print("Sample features (first 5 symptoms):")
print(X.iloc[:3, :5])

Feature matrix shape: (4920, 131)
Sample features (first 5 symptoms):
   abdominal_pain  abnormal_menstruation  acidity  acute_liver_failure  \
0               0                      0        0                    0   
1               0                      0        0                    0   
2               0                      0        0                    0   

   altered_sensorium  
0                  0  
1                  0  
2                  0  


In [3]:
from sklearn.model_selection import train_test_split

# Encode disease names to numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Disease classes:", le.classes_[:10], "...")
print("Encoded labels sample:", y_encoded[:5])

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Disease classes: ['(vertigo) Paroymsal  Positional Vertigo' 'AIDS' 'Acne'
 'Alcoholic hepatitis' 'Allergy' 'Arthritis' 'Bronchial Asthma'
 'Cervical spondylosis' 'Chicken pox' 'Chronic cholestasis'] ...
Encoded labels sample: [15 15 15 15 15]

Training samples: 3936
Testing samples: 984


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import time

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    train_time = round(time.time() - start, 2)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = {
        "accuracy": round(acc * 100, 2),
        "train_time": train_time
    }
    print(f"  Accuracy: {acc*100:.2f}% | Time: {train_time}s")

print("\n--- RESULTS ---")
for name, r in results.items():
    print(f"{name}: {r['accuracy']}% accuracy in {r['train_time']}s")

Training Logistic Regression...
  Accuracy: 100.00% | Time: 0.15s
Training Random Forest...
  Accuracy: 100.00% | Time: 0.23s
Training XGBoost...
  Accuracy: 100.00% | Time: 0.88s

--- RESULTS ---
Logistic Regression: 100.0% accuracy in 0.15s
Random Forest: 100.0% accuracy in 0.23s
XGBoost: 100.0% accuracy in 0.88s


In [6]:
import pickle
import os

# Find best model
best_name = max(results, key=lambda x: results[x]['accuracy'])
best_model = models[best_name]
best_acc = results[best_name]['accuracy']

print(f"Best model: {best_name} with {best_acc}% accuracy")

# Save model + label encoder + symptom list
os.makedirs('../models', exist_ok=True)

with open('../models/symptom_classifier.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'label_encoder': le,
        'symptoms': all_symptoms,
        'model_name': best_name,
        'accuracy': best_acc
    }, f)

print("Model saved to ../models/symptom_classifier.pkl")

Best model: Logistic Regression with 100.0% accuracy
Model saved to ../models/symptom_classifier.pkl


In [7]:
# Test prediction
with open('../models/symptom_classifier.pkl', 'rb') as f:
    saved = pickle.load(f)

model = saved['model']
le = saved['label_encoder']
symptoms_list = saved['symptoms']

def predict_disease(patient_symptoms: list) -> list:
    """
    Input: list of symptoms
    Output: top 3 predicted diseases with confidence
    """
    # Create feature vector
    X_input = pd.DataFrame(0, index=[0], columns=symptoms_list)
    for s in patient_symptoms:
        s = s.strip().lower().replace(' ', '_')
        if s in X_input.columns:
            X_input[s] = 1
    
    # Get probabilities
    probs = model.predict_proba(X_input)[0]
    top3_idx = probs.argsort()[-3:][::-1]
    
    results = []
    for idx in top3_idx:
        results.append({
            "disease": le.classes_[idx],
            "confidence": round(float(probs[idx]) * 100, 1)
        })
    return results

# Test with real symptoms
test_symptoms = ["fever", "headache", "fatigue", "high_fever"]
predictions = predict_disease(test_symptoms)

print("Input symptoms:", test_symptoms)
print("\nPredictions:")
for p in predictions:
    print(f"  {p['disease']}: {p['confidence']}%")

Input symptoms: ['fever', 'headache', 'fatigue', 'high_fever']

Predictions:
  Bronchial Asthma: 10.7%
  AIDS: 9.9%
  Paralysis (brain hemorrhage): 8.7%
